# Session 0: Prerequisites

Welcome to the PydanticAI course! This notebook introduces the prerequisite components you'll need to understand before working with PydanticAI:
1. **asyncio** - Python's library for asynchronous programming. Working with agents means many API calls, which means concurrency.
2. **Pydantic** - Data validation using Python type annotations. Pydantic AI is built using these components.

In this session you'll learn enough about each of these packages to not slow you down for the rest of this course.


## Part 1: Introduction to asyncio
The `asyncio` library allows you to write concurrent code using the `async`/`await` syntax. It is similar, but subtly different from multi-threading and multi-processing.

Imagine you have a task that does some computation for 1 second, waits for 30 seconds, and then does 1 second more of computation, and another task that does 20 seconds of computation. You could run these tasks in sequence, which would take 32 + 20 = 52 seconds, or you could run the second while you're waiting for the first one between computation, which would only take 32 seconds in total.

As a more practical example, imagine you have a Streamlit user interface that includes a chatbot. Sending calls to LLM APIs require a tiny amount of computation, then a wait of several seconds while OpenAI run their model, and then another tiny bit of computation to handle the response. If you handled things synchronously, your UI would be unusable for several seconds while you wait for an API response.

So how do you use `asyncio`? We recommend priming your brain with the [asyncio conceptual overview](https://docs.python.org/3/howto/a-conceptual-overview-of-asyncio.html) in the Python docs. 

### Key Concepts

- **The event loop**: The secret sauce of asyncio is the event loop - the conductor of operations, and its managed for you. It hands control to jobs and waits for them to hand control back, so relies on jobs being co-operative and not hogging control.
- **coroutine functions**: Create using the `async def` keywords. Rather than running their code and returning a result they return **coroutine objects**.
- **coroutine objects**: A set of instructions to be run. They can be turned into **tasks** using `asyncio.create_task()` or run with `asyncio.run()`.
- **task**: A coroutine object attached to an event loop. This is done by adding a callback to the task, not the task itself. Because of this separation they must be shielded with the `await` keyword to make sure they aren't garbage cleaned before they can be used.
- **await**: behaves in two ways, depending on whether it is used with a **task** or a **coroutine**. The keyword can only be used within a **coroutine function**. 
    - Awaiting a task passes control back to the event loop which will decide when to run the task. 
    - Awaiting a coroutine runs the coroutine immediately, blocking execution until the coroutine is finished - functionally the same as synchronous (normal) Python operation. Remember you can use `asyncio.create_task()` to turn coroutines into tasks!

```ascii
              async def
                  |
              (defines)
                  |
                  v
        [ coroutine function ]
                  |
              (returns)
                  |
                  v
         [ coroutine object ]
                  |
      +-----------+-----------------------------+
      |                                         |
  [ await ]                           (asyncio.create_task)
      |                                         |
      |                                         v
      v                                    +--------+       PASS CONTROL      +------------+
(Execute Now)                              |  TASK  |------------------------>| EVENT LOOP |
                                           +--------+    (attach callback)    +------------+
"Functionally                                   ^
 Synchronous"                                   |
                                                |
                                            [ await ]  <-- (shield)
                                                |
                                      (attempts to collect)
                                                |
                                      [ garbage collector ]
```

## An important note about async in notebooks
Since Jupyter notebooks already run in an async loop - which is how you can trigger cells to run code and then go do other things in the UI, asyncio will behave strangely here. You can't call `asyncio.run()`, and will get an Exception if you try. You'll also notice that you can call `await` seemingly outside of coroutine functions.

The solution is simple: Replace `asyncio.run(my_async_function())` with `await my_async_function()`. This will result in your coroutine running in the notebook's existing event loop. Just remember to swap back to `asyncio.run()` when working in .py files outside of notebooks. (a second hacky solution is to use the nest_asyncio package, which allows nested event loops, but it is not stable and is now an archived package).

With that in mind, we'll show both Jupyter notebook and .py implementations of a few examples.

In [1]:
import asyncio
import time

### Example 1: Sequential vs Asynchronous Execution

Let's compare sequential execution (blocking) with asynchronous execution (non-blocking).


In [2]:
# Sequential (blocking) version
def fetch_data_sequential(task_id: int, delay: float) -> str:
    """Simulates a long-running I/O operation using time.sleep"""
    print(f"Task {task_id}: Starting (will take {delay}s)")
    time.sleep(delay)  # Blocks the entire program
    print(f"Task {task_id}: Completed")
    return f"Result from task {task_id}"


# Asynchronous (non-blocking) version
async def fetch_data_async(task_id: int, delay: float) -> str:
    """Simulates a long-running I/O operation using asyncio.sleep"""
    print(f"Task {task_id}: Starting (will take {delay}s)")
    await asyncio.sleep(delay)  # Yields control to other tasks
    print(f"Task {task_id}: Completed")
    return f"Result from task {task_id}"

In [3]:
result = fetch_data_sequential(1, 1.0)
print(f"Result of fetch_data_sequential(1, 1.0): {result}")

result = fetch_data_async(1, 1.0)
print(f"Result of fetch_data_async(1, 1.0): {result}")

Task 1: Starting (will take 1.0s)
Task 1: Completed
Result of fetch_data_sequential(1, 1.0): Result from task 1
Result of fetch_data_async(1, 1.0): <coroutine object fetch_data_async at 0x1089c05e0>


You can see the output of the sequential version is the expected string, but the async version is a coroutine object. You may also see some RuntimeWarnings because we didn't run our async function with the await keyword, and never turned it into a task - meaning it may cause problems in the future.

Now let's run both versions 3 times and compare the execution time:


In [4]:
# Sequential execution - tasks run one after another
print("=== Sequential Execution ===")
start = time.time()
results_seq = [
    fetch_data_sequential(1, 1.0),
    fetch_data_sequential(2, 1.0),
    fetch_data_sequential(3, 1.0),
]
end = time.time()
print(f"Sequential took: {end - start:.2f} seconds\n")

# Asynchronous execution - tasks run concurrently
print("=== Asynchronous Execution ===")


async def run_async():
    start = time.time()
    results = await asyncio.gather(
        fetch_data_async(1, 1.0),
        fetch_data_async(2, 1.0),
        fetch_data_async(3, 1.0),
    )
    end = time.time()
    print(f"Asynchronous took: {end - start:.2f} seconds\n")
    return results


_ = await run_async()  # Should be asyncio.run(run_async()) in .py

=== Sequential Execution ===
Task 1: Starting (will take 1.0s)
Task 1: Completed
Task 2: Starting (will take 1.0s)
Task 2: Completed
Task 3: Starting (will take 1.0s)
Task 3: Completed
Sequential took: 3.01 seconds

=== Asynchronous Execution ===
Task 1: Starting (will take 1.0s)
Task 2: Starting (will take 1.0s)
Task 3: Starting (will take 1.0s)
Task 1: Completed
Task 2: Completed
Task 3: Completed
Asynchronous took: 1.00 seconds



The .py equivalent can by run with `uv run python session_0_example_1.py`.

So what is happening here? The first version of the code `fetch_data_sequential` hogs the main thread, meaning we have to wait the full 1 second for each task to complete, and in total we wait 3 seconds for the 3 tasks. In the second version, `fetch_data_async` we replace `time.sleep` with `asyncio.sleep`. What is different? Why do we need to use `asyncio`'s sleep function?

Broadly, this version of sleep yields control back to the main thread, using the `yield` keyword, but the original version doesn't. We could go down a rabbit hole here, so those who are interested should look at [this](https://docs.python.org/3/howto/a-conceptual-overview-of-asyncio.html#a-conceptual-overview-of-asyncio) link to the docs.

You can build asynchronous functions from other asynchronous functions using all the logic you normally would in Python. Beware not to include any long running functions that aren't asynchronous (such as doing some heavy computation or a bare pd.read_csv) in these functions, as they would then hog the event loop and slow everything down. A rule of thumb is that all async functions should take <100ms to run. The async debugger will actually flag those that take longer as issues.

### Example 2: asyncio.create_task - Creating and Running Several Tasks

In Example 1 we used `asyncio.gather()` to run coroutines concurrently (it can also run tasks). Another approach is `asyncio.create_task()`, which schedules a coroutine to run on the event loop and returns a Task object immediately.

Here we create several tasks in a coroutine function. The tasks are scheduled, but won't be run until the event loop hands them control. Even though they appear to run sequentially, they are non-blocking so the total run time is ~1 second rather than 3.

In [ ]:
# create_task schedules each coroutine on the event loop - all start running immediately
# In a notebook we use await directly (event loop is already running)
async def run_create_task_example():
    print("=== asyncio.create_task Example ===")
    start = time.time()

    # Create all tasks - they start running as soon as created
    task1 = asyncio.create_task(fetch_data_async(1, 1.0))
    task2 = asyncio.create_task(fetch_data_async(2, 1.0))
    task3 = asyncio.create_task(fetch_data_async(3, 1.0))

    # Await each task to get results (they run concurrently, so total ~1s not 3s)
    result1 = await task1
    result2 = await task2
    result3 = await task3

    end = time.time()
    print(f"Completed in {end - start:.2f} seconds")
    print(f"Results: {result1}, {result2}, {result3}")


_ = await run_create_task_example()  # Should be asyncio.run(run_create_task_example()) in .py

The .py equivalent can be run with `uv run python session_0_example_2.py`.

**Important**: Always `await` your tasks before they go out of scope. Task objects need to be referenced; otherwise the garbage collector may discard them before they complete. The `await` acts as a "shield" that keeps them alive until they finish.

### Connection to PydanticAI

All you really need to know for PydanticAI is that it includes asynchronous methods, and you should use the await keyword if you want your code to run in the order you think it should! For example:

```python
from pydantic_ai import Agent

agent = Agent(...)

# All agent operations are async
result = await agent.run("Your prompt here")

# You can run multiple agent calls concurrently
results = await asyncio.gather(
    agent.run("Prompt 1"),
    agent.run("Prompt 2"),
    agent.run("Prompt 3"),
)

# You can also run synchronously with - noting that this won't work when called from notebooks!
result = agent.run_sync("Your prompt here")
```

The control flow patterns you've learned (loops, conditionals, error handling) work the same way with PydanticAI agents, allowing you to build sophisticated AI workflows.

## Part 2: Introduction to Pydantic

The second important concept we'll need to know is Pydantic (not to be confused with Pydantic AI). Pydantic is a data validation library that uses Python type annotations. It's the foundation that PydanticAI is built upon.

### Key Concepts

- **`BaseModel`**: The base class for all Pydantic models
- **Type annotations**: Define the expected types for model fields
- **Automatic validation**: Pydantic validates data based on type hints
- **Serialization**: Easy conversion to/from JSON and dictionaries - create for sending data to/from APIs.


In [5]:
from typing import Optional

from pydantic import BaseModel, Field, field_validator

### Example 1: Basic Pydantic Model

Let's start with a simple model representing a user. Create a model by defining a class that inherits from BaseModel. Then add class level fields, in this case name, age, email, and is_active. These can also be given types using the `:` notation - so the name field is of the string type, and given default values using the `=` notation - so is_active will default to False.

You can then create instances of your new model by instantiating a new object of that class. If you include invalid arguments, you'll get a runtime error. Usefully, your IDE should flag invalid inputs too!

In [8]:
class User(BaseModel):
    """A simple user model"""

    name: str
    age: int
    email: str
    is_active: bool = True  # Default value


# Create an instance - validation happens automatically
user = User(name="Alice", age=30, email="alice@example.com")
print(f"User: {user.name}, Age: {user.age}, Email: {user.email}")
print(f"Model as dict: {user.model_dump()}")
print(f"Model as JSON: {user.model_dump_json()}")

# Try with invalid data (uncomment to see validation error)
# invalid_user = User(name="Bob", age="thirty", email="not-an-email")

User: Alice, Age: 30, Email: alice@example.com
Model as dict: {'name': 'Alice', 'age': 30, 'email': 'alice@example.com', 'is_active': True}
Model as JSON: {"name":"Alice","age":30,"email":"alice@example.com","is_active":true}


### Example 2: Model with Field Validation

Pydantic allows you to add more complex validation rules using `Field` and validators:


In [ ]:
class Product(BaseModel):
    """A product model with validation"""

    name: str = Field(min_length=1, max_length=100)
    price: float = Field(gt=0, description="Price must be positive")
    quantity: int = Field(ge=0, description="Quantity cannot be negative")
    category: Optional[str] = None

    @field_validator("name")
    @classmethod
    def validate_name(cls, v: str) -> str:
        """Custom validator for name"""
        if not v.strip():
            raise ValueError("Name cannot be empty or whitespace")
        return v.strip().title()  # Capitalize each word


# Valid product
product = Product(name="laptop computer", price=999.99, quantity=5)
print(f"Product: {product.name}, Price: ${product.price}, Quantity: {product.quantity}")

# Invalid product (uncomment to see validation error)
# invalid_product = Product(name="  ", price=-10, quantity=-1)

### Example 3: Nested Models

Pydantic models can contain other Pydantic models:


In [9]:
class Address(BaseModel):
    street: str
    city: str
    zip_code: str


class Person(BaseModel):
    name: str
    age: int
    address: Address  # Nested model


# Create a person with nested address
person = Person(
    name="John Doe",
    age=35,
    address=Address(street="123 Main St", city="New York", zip_code="10001"),
)

print(f"Person: {person.name}")
print(f"Address: {person.address.street}, {person.address.city}")
print(f"Full model: {person.model_dump()}")

Person: John Doe
Address: 123 Main St, New York
Full model: {'name': 'John Doe', 'age': 35, 'address': {'street': '123 Main St', 'city': 'New York', 'zip_code': '10001'}}


### Exercise 3: Create a Pydantic Model

Create a `Book` model with:
- `title` (string, required, min length 1)
- `author` (string, required)
- `isbn` (string, optional)
- `pages` (integer, must be > 0)
- `published_year` (integer, between 1000 and current year)

Add a validator that ensures the ISBN (if provided) is exactly 13 characters.


In [11]:
# Your solution here
class Book(BaseModel):
    # TODO: Define the fields with appropriate types and Field constraints
    # Hint: Use Field() for constraints, @field_validator for custom validation
    title: str = Field(min_length=1)
    author: str
    isbn: Optional[str] = Field(default = None)
    pages: int = Field(ge=1, description="Number must be greater than 1")
    published_year: int = Field(ge=1000, le=2026)


# Test your solution with assert statements
# Uncomment these tests once you've implemented your Book model

# Test 1: Valid book with all fields
book1 = Book(
    title="Python Programming",
    author="Jane Smith",
    isbn="9780123456789",
    pages=500,
    published_year=2023
)
assert book1.title == "Python Programming"
assert book1.author == "Jane Smith"
assert book1.isbn == "9780123456789"
assert book1.pages == 500
assert book1.published_year == 2023
print("✓ Test 1 passed: Valid book with all fields")

# Test 2: Valid book without ISBN (optional field)
book2 = Book(
    title="Data Science Handbook",
    author="John Doe",
    pages=300,
    published_year=2022
)
assert book2.isbn is None
print("✓ Test 2 passed: Valid book without ISBN")

# Test 3: Validation should fail for invalid data
# Uncomment to test that validation works:
from pydantic import ValidationError
try:
    invalid_book = Book(title="", author="Test", pages=100, published_year=2023)
    assert False, "Should have raised ValidationError for empty title"
except ValidationError:
    print("✓ Test 3 passed: Validation correctly rejects empty title")

try:
    invalid_book = Book(title="Test", author="Test", pages=-1, published_year=2023)
    assert False, "Should have raised ValidationError for negative pages"
except ValidationError:
    print("✓ Test 4 passed: Validation correctly rejects negative pages")

✓ Test 1 passed: Valid book with all fields
✓ Test 2 passed: Valid book without ISBN
✓ Test 3 passed: Validation correctly rejects empty title
✓ Test 4 passed: Validation correctly rejects negative pages


### Exercise 4: Custom Validators with the Book Model

Now that you've created the `Book` model in Exercise 3, let's extend it with a custom validator. Custom validators allow you to implement complex validation logic that goes beyond simple type checking and field constraints.

**Task**: Add a custom validator to your `Book` model that validates the `isbn` field. The validator should:
- If the book has an ISBN, validate that the ISBN format is correct (should be 13 digits, optionally with hyphens)

**Hint**: You can use `@field_validator` decorator with the `mode='before'` or `mode='after'` parameter. The `mode='after'` is the default and receives the validated value. Remember to handle the case where `isbn` might be `None` since it's an optional field.


In [15]:
# Extend your Book model from Exercise 3 with custom validators
import re
from typing import Optional

from pydantic import BaseModel, Field, field_validator

# TODO: Add a custom validator for the isbn field
class Book(BaseModel):
    # TODO: Define the fields with appropriate types and Field constraints
    # Hint: Use Field() for constraints, @field_validator for custom validation
    title: str = Field(min_length=1)
    author: str
    isbn: Optional[str] = Field(default = None)
    pages: int = Field(ge=1, description="Number must be greater than 1")
    published_year: int = Field(ge=1000, le=2026)

    # Hint: Use @field_validator to validate the ISBN format when provided
    @field_validator('isbn')
    @classmethod
    def validate_isbn(cls, value):
        if value is None:
            return value

        # Remove hyphens before checking the digit count
        cleaned = value.replace('-', '')

        if not cleaned.isdigit():
            raise ValueError('ISBN must contain only digits (hyphens allowed)')

        if len(cleaned) != 13:
            raise ValueError('ISBN must be exactly 13 digits')

        return value  # or `return cleaned` if you want to store the normalized form


# Test your extended Book model
valid_book = Book(
    title="Python Programming",
    author="Jane Smith",
    isbn="978-0-12-345678-9",
    pages=500,
    published_year=2023
)
print("Valid book:", valid_book)

# Test invalid cases (uncomment to see validation errors)
# invalid_isbn = Book(title="Test", author="Author", isbn="123", pages=100, published_year=2023)
valid_no_isbn = Book(title="Test", author="Author", pages=100, published_year=2023)  # ISBN is optional

Valid book: title='Python Programming' author='Jane Smith' isbn='978-0-12-345678-9' pages=500 published_year=2023


### Example 4: Using Pydantic with OpenAI API for Structured Output

One of the most powerful features of Pydantic is using it to structure LLM responses. OpenAI's API supports structured outputs via JSON schema mode. Here's how to use a Pydantic model with the OpenAI API:

**Before running this example**, you'll need to:
1. Get an OpenAI API key - [https://platform.openai.com/api-keys](https://platform.openai.com/api-keys)
2. Create a `.env` file in your project root directory and add your API key `OPENAI_API_KEY=<key-here>` following the .env.example file 

In this example we'll make a simple [sentiment analysis](https://en.wikipedia.org/wiki/Sentiment_analysis) model. We'll define the exact form of the output, and use OpenAI's inbuilt functionality to enforce the schema.

We'll use `pydantic-settings` to load configuration (such as your OpenAI API key) from environment variables and the `.env` file. That keeps secrets out of code and makes it easy to switch between environments that may require different configurations (for example local-dev with local resources vs a staging environment and cloud resources). You define a settings class, and `pydantic-settings` fills it from `os.environ` and `.env` automatically.

In [16]:
from openai import OpenAI
from pydantic_settings import BaseSettings, SettingsConfigDict


class Settings(BaseSettings):
    model_config = SettingsConfigDict(env_file=".env")
    openai_api_key: str
    open_ai_default_model: str = "openai:gpt-5-nano"


settings = Settings()

In [17]:
# First, let's define a Pydantic model for structured output
class SentimentAnalysis(BaseModel):
    """Model for sentiment analysis results"""

    sentiment: str = Field(description="The sentiment: 'positive', 'negative', or 'neutral'")
    score: float = Field(ge=0.0, le=1.0, description="Confidence score between 0 and 1")
    reasoning: str = Field(description="Brief explanation of why this sentiment was chosen")
    keywords: list[str] = Field(
        default_factory=list,
        description="Key words or phrases that influenced the analysis",
    )

    @field_validator("sentiment")
    @classmethod
    def validate_sentiment(cls, v: str) -> str:
        """Ensure sentiment is one of the valid values"""
        valid = {"positive", "negative", "neutral"}
        if v.lower() not in valid:
            raise ValueError(f"Sentiment must be one of {valid}")
        return v.lower()


def print_sentiment_analysis(result):
    print("Sentiment Analysis Result:")
    print(f"Sentiment: {result.sentiment}")
    print(f"Score: {result.score:.2f}")
    print(f"Reasoning: {result.reasoning}")
    print(f"Keywords: {result.keywords}")


# Now let's use this model with OpenAI API
def analyze_sentiment_with_openai(text: str) -> SentimentAnalysis:
    """Use OpenAI API with Pydantic model for structured output"""

    # Create the OpenAI client
    client = OpenAI(api_key=settings.openai_api_key)

    # Call OpenAI with structured output mode
    response = client.beta.chat.completions.parse(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "system",
                "content": "You are a sentiment analysis expert. Analyze the sentiment of the given text.",
            },
            {
                "role": "user",
                "content": f"Analyze the sentiment of this text: {text}",
            },
        ],
        response_format=SentimentAnalysis,  # OpenAI supports Pydantic models directly!
        temperature=0.7,
    )

    # The response is already a validated Pydantic model instance!
    parsed = response.choices[0].message.parsed
    if parsed is None:
        raise ValueError("Failed to parse response as SentimentAnalysis")
    return parsed


result = analyze_sentiment_with_openai("I absolutely love this new Python library! It's amazing!")
print_sentiment_analysis(result)

result = analyze_sentiment_with_openai(r"I **absolutely** love this new Python library! It's so amazing! /s")
print_sentiment_analysis(result)

/var/folders/5b/x8050hc55gqdtp1l61zgjnzh0000gp/T/ipykernel_97696/3461373408.py:62: RuntimeWarning: coroutine 'fetch_data_async' was never awaited
  result = analyze_sentiment_with_openai("I absolutely love this new Python library! It's amazing!")


Sentiment Analysis Result:
Sentiment: positive
Score: 0.95
Reasoning: The text expresses strong enthusiasm and positive feelings towards the Python library, indicated by the words 'absolutely love' and 'amazing'.
Keywords: ['love', 'amazing', 'new Python library']
Sentiment Analysis Result:
Sentiment: negative
Score: 0.85
Reasoning: The use of '/s' indicates that the statement is sarcastic. Despite the positive words like 'love' and 'amazing', the sarcasm signals that the true sentiment is negative.
Keywords: ['absolutely', 'love', 'amazing', 'sarcasm']


Just a note that identifying sarcasm was once a holy grail to me of sentiment analysis - something really hard to do with traditional NLP approaches and the classic weak point of the wonderfully simple approaches like [VADER](https://github.com/cjhutto/vaderSentiment).





### Connection to PydanticAI

In PydanticAI, you'll define models like this to structure your AI agent's responses:

```python
from pydantic_ai import Agent

class AnalysisResult(BaseModel):
    sentiment: str
    score: float
    reasoning: str

agent = Agent(
    'openai:gpt-4',
    output_type=AnalysisResult  # PydanticAI ensures the LLM returns this structure
)

result = await agent.run("Analyze this text: 'I love Python!'")
# result.data will be an AnalysisResult instance with validated fields
```

The validation ensures that LLM outputs conform to your expected structure, making your AI applications more reliable and type-safe.


## Summary

You've now learned:

1. **asyncio fundamentals**:
   - How to define async functions with `async def`
   - How to await async operations
   - How to run multiple tasks concurrently with `asyncio.gather()`

2. **Pydantic fundamentals**:
   - How to create models using `BaseModel`
   - Field validation with `Field` constraints
   - Custom validators with `@field_validator`
   - Nested models
   - How these concepts apply to PydanticAI

You're now ready to dive into PydanticAI in session 1!


# Answers and Hints


In [ ]:
# Example structure (replace with your Book model from Exercise 3):
# class Book(BaseModel):
#     title: str = Field(min_length=1)
#     author: str
#     isbn: Optional[str] = None
#     pages: int = Field(gt=0)
#     published_year: int = Field(ge=1000, le=2024)
#
#     @field_validator('isbn')
#     @classmethod
#     def validate_isbn(cls, v: Optional[str]) -> Optional[str]:
#         """Validate ISBN format if provided"""
#         if v is None:
#             return v
#         # Remove hyphens and check if it's 13 digits
#         isbn_clean = re.sub(r'-', '', v)
#         if not (isbn_clean.isdigit() and len(isbn_clean) == 13):
#             raise ValueError("ISBN must be exactly 13 digits (hyphens optional)")
#         return isbn_clean  # Return cleaned version